In [8]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((32,32)),  #이미지 크기 통일
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

train_dataset= datasets.ImageFolder(root='./data/horse-or-human/train', transform=transform)
test_dataset= datasets.ImageFolder(root='./data/horse-or-human/test', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=5, shuffle=False)




In [7]:
img,label = next (iter(train_loader))
img.size(), label

(torch.Size([5, 3, 32, 32]), tensor([1, 1, 1, 0, 1]))

In [9]:
#######실행1

import torch.nn as nn
class BasicBlock(nn.Module):
    def __init__ (self, int_channels, out_channels, hidden_dim):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(int_channels, hidden_dim, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(hidden_dim, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
    def forward(self, x) : 
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        out = self.pool(x)
        return out
    

#######실행2

import torch
class CNN(nn.Module):
    def __init__ (self, num_class):
        super(CNN, self).__init__()
        
        #상기 셀에 BasicBlock에 정의된 순서 int_channels, out_channels, hidden_dim  
        self.block1 = BasicBlock(3, 64, 64)   # c c p :  (3,32,32) --> (64, 32, 32) --> (64, 32,32) --> (64, 16, 16)
        self.block2 = BasicBlock(64, 128, 128) # --> (128, 16, 16) --> (128, 16, 16) --> (128, 8, 8)

        # 분류기
        self.fc1 = nn.Linear ( 128*8*8,2048)     
        self.fc2 = nn.Linear (2048, 256)
        self.fc3 = nn.Linear (256, num_class)
        self.relu = nn.ReLU()

    def forward (self, x):
        x= self.block1(x)
        x= self.block2(x)
        # (-1,128*8*)
        x=torch.flatten(x,start_dim=1) # 1d로 펴주기. 데이터갯수는 두고..???
        x= self.relu (self.fc1(x)) #왜 relu를 또하지??????????????????????????
        x= self.relu (self.fc2(x))
        out= self.fc3(x)
        return out 


#######실행3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN(2)
model.to(device)


#######실행4

# 하이퍼 파라미터
from torch.optim import Adam 
lr = 1e-3
optimizer = Adam(model.parameters(), lr=lr)
epochs = 20


from tqdm import tqdm 
# 학습루프
for epoch in range(epochs):
    tqdm_obj= tqdm(train_loader, desc=f'epoch: {epoch+1}/{epochs}')
    for data, label in tqdm_obj:
        optimizer.zero_grad()
        preds = model (data.to(device))
        loss = nn.CrossEntropyLoss()(preds, label.to(device))
        loss.backward()
        optimizer.step()
        
        tqdm_obj.set_postfix(loss=loss.item())

torch.save(model.state_dict(), 'hm.pth')  #torch에는 모델 저장 기능이 있음 / 가중치만 저장되어있음./ 모델구조는 저장되어 있지 않음. (모델+가중치 저장되는 것도 있음. 용량 많이 차지함)
        


epoch: 20/20: 100%|██████████| 206/206 [00:41<00:00,  4.99it/s, loss=2.74e-5]


In [10]:
#######실행5
# 평가
model.load_state_dict(torch.load('hm.pth', map_location=device, weights_only=True))

# 예측
# 평가 루프
# test_loader가 이미 정의되어 있다고 가정
model.eval()  # 평가 모드로 전환 (dropout, batchnorm 등 비활성화)
total_loss = 0.0
total_correct = 0
total_samples = 0

criterion = nn.CrossEntropyLoss()
with torch.no_grad():  # 그래디언트 계산 비활성화
    for data, label in tqdm(test_loader, desc="Evaluating"):
        data, label = data.to(device), label.to(device)
        preds = model(data)
        loss = criterion(preds, label)
        total_loss += loss.item() * data.size(0)  # 배치 손실 합산
        total_correct += (preds.argmax(dim=1) == label).sum().item()
        total_samples += data.size(0)

avg_loss = total_loss / total_samples
accuracy = total_correct / total_samples

print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.4f}")


Evaluating: 100%|██████████| 52/52 [00:04<00:00, 11.75it/s]

Test Loss: 1.6972, Test Accuracy: 0.9141
